# Exécution de différentes configurations de LanSDP

Ce notebook permet d'exécuter et de comparer les performances du modèle `LanSDP` avec différentes configurations pour les paramètres `l_max` et `chordal_stride`.

Les paramètres `l_max` (couche maximale pour la relaxation McCormick) et `chordal_stride` (taille des cliques pour la décomposition chordale) ont un impact significatif sur la taille du problème SDP et la qualité de la relaxation. Ce notebook vise à explorer cet impact.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import tempfile
import shutil

# Ajouter le répertoire racine du projet au PYTHONPATH
project_root = os.path.abspath(os.path.join(os.getcwd(), ".")) # Assurez-vous que le notebook est à la racine ou ajustez le chemin
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.networks.network import ReLUNN
from src.solve.sdp_solve.SDPmodels.Lan_SDP import LanSDP
from src.fastsdp_tools import get_project_path

# --- Constantes --- 
DATA_INDEX = 0
EPSILON = 0.5
NORM = "Linf"

NETWORK_PATH = "data/models/blob_adv_blob_nn_4x10.pt"
DATASET_PATH = "data/datasets/blob_dataset.pth"
BOUNDS_PATH = "data/bounds/blob_4x10-0.5_linf.csv"

# --- Fonctions utilitaires pour le chargement des données --- 

def _load_network() -> ReLUNN:
    return ReLUNN.from_pth(get_project_path(NETWORK_PATH), bb_beta_crown=False)

def _load_sample(data_index: int):
    raw = torch.load(get_project_path(DATASET_PATH), weights_only=False)
    dataset = raw["dataset"]
    x, label = dataset[data_index]
    x_t = x.view(-1).float()
    ytrue = int(label.item())
    return x_t, ytrue

def _load_bounds(network: ReLUNN, data_index: int):
    """Charge L, U depuis le CSV de bornes pré-calculées."""
    df = pd.read_csv(get_project_path(BOUNDS_PATH))
    row = df[df["data_index"] == data_index].iloc[0]
    K = network.K
    n = network.n
    L = []
    U = []
    for k in range(K + 1):
        L.append([float(row[f"LB_Layer_{k}_Neuron_{j}"]) for j in range(n[k])])
        U.append([float(row[f"UB_Layer_{k}_Neuron_{j}"]) for j in range(n[k])])
    return L, U

# --- Fonction pour exécuter LanSDP avec des configurations spécifiques --- 

def run_lan_sdp_config(network, x, ytrue, ytargets, L, U, ytarget, l_max, chordal_stride, cuts=None):
    if cuts is None:
        cuts = []
        
    # Créer un répertoire temporaire pour chaque exécution
    temp_dir = tempfile.mkdtemp()
    
    try:
        solver = LanSDP(
            network=network,
            epsilon=EPSILON,
            x=x,
            ytrue=ytrue,
            ytargets=ytargets,
            ytarget=ytarget,
            L=L,
            U=U,
            cuts=cuts,
            MATRIX_BY_LAYERS=True, # Toujours True pour tester chordal_stride
            l_max=l_max,
            chordal_stride=chordal_stride,
            RLT_props=[0.2], # Valeur par défaut, peut être ajustée
            use_active_neurons=False,
            use_inactive_neurons=False,
            keep_penultimate_actives=True,
            norm=NORM,
            bounds_method="IBP",
            network_name="blob_4x10",
            dataset_name="blob",
            data_index=DATA_INDEX,
            INPUT_IN_VARIABLES=True,
            folder_name=temp_dir # Utiliser le répertoire temporaire
        )
        
        solver.solve()
        df = solver.benchmark_dataframe
        if df is not None and "optimal_value" in df.columns:
            return float(df["optimal_value"].iloc[-1])
        else:
            return np.nan
    finally:
        # Nettoyer le répertoire temporaire
        shutil.rmtree(temp_dir)


print("Setup complet.")

ModuleNotFoundError: No module named 'torch'

: 

## Exécution des tests

Nous allons maintenant exécuter `LanSDP` pour différentes combinaisons de `l_max` et `chordal_stride`.

In [ ]:
# --- Chargement des données de base --- 
network = _load_network()
x, ytrue = _load_sample(DATA_INDEX)
L, U = _load_bounds(network, DATA_INDEX)

n_out = network.n[network.K]
ytargets = [j for j in range(n_out) if j != ytrue]

# Choisir un ytarget pour la démonstration (le premier disponible)
if not ytargets:
    raise ValueError("Aucun ytarget disponible. Le réseau est trivialement vérifié ou le data_index est incorrect.")
selected_ytarget = ytargets[0]

print(f"Réseau chargé : {network.name}")
print(f"Exemple de données : data_index={DATA_INDEX}, ytrue={ytrue}, ytarget={selected_ytarget}")

# --- Définition des configurations à tester --- 
configurations = []
l_max_values = [network.K, 3, 1] # network.K = pas de pruning, 3 = pruning après couche 3, 1 = pruning après couche 1
chordal_stride_values = [1, 2] # 1 = paires consécutives, 2 = cliques de 3 couches

for l_max in l_max_values:
    for stride in chordal_stride_values:
        configurations.append({"l_max": l_max, "chordal_stride": stride})

results = []

print("\nLancement des exécutions LanSDP...")
for config in configurations:
    l_max = config["l_max"]
    chordal_stride = config["chordal_stride"]
    print(f"\nExécution avec l_max={l_max}, chordal_stride={chordal_stride}...")
    
    optimal_value = run_lan_sdp_config(
        network, x, ytrue, ytargets, L, U, selected_ytarget, l_max, chordal_stride
    )
    
    results.append({
        "l_max": l_max,
        "chordal_stride": chordal_stride,
        "optimal_value": optimal_value
    })
    print(f"  Valeur optimale : {optimal_value:.6f}")

results_df = pd.DataFrame(results)

print("\n--- Résultats récapitulatifs ---")
print(results_df)

print("\nAnalyse terminée.")